# **DEFINISI FUNGSI MODULAR**

---



## Setup MNIST

In [11]:
def setup_mnist_environment(model_name):
    """
    Menangani mounting drive, setup path, dan inisialisasi DataLoaders.
    """
    from google.colab import drive
    import os
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader

    # 1. Mount Drive
    drive.mount('/content/drive', force_remount=True)

    # 2. Setup Paths
    model_dir = '/content/drive/MyDrive/model_modular'
    os.makedirs(model_dir, exist_ok=True)
    path = os.path.join(model_dir, model_name)

    # 3. DataLoaders
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)) # Standard normalization for MNIST
    ])

    train_set = datasets.MNIST(root='./data_mnist', train=True, download=True, transform=transform)
    test_set = datasets.MNIST(root='./data_mnist', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=1000, shuffle=False)

    return path, train_loader, test_loader

## Load or Training Model & Eval Model

In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

def evaluate_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def load_and_prepare_model(model_class, model_path, train_loader, test_loader, device, num_epochs=10, lr=0.001):
    model = model_class().to(device)
    history = {'loss': [], 'accuracy': []}

    if os.path.exists(model_path):
        print(f"✅ Loading existing model from {model_path}...")
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        history = checkpoint.get('history', history)
        print(f"✅ Model loaded. Previous accuracy: {history['accuracy'][-1]:.2f}%" if history['accuracy'] else "✅ Model loaded.")
    else:
        print(f"❌ No model found at {model_path}. Starting training...")
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(num_epochs):
            model.train()
            running_loss = 0.0
            for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            acc = evaluate_accuracy(model, test_loader, device)
            avg_loss = running_loss / len(train_loader)
            history['loss'].append(avg_loss)
            history['accuracy'].append(acc)
            print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Accuracy = {acc:.2f}%")

        print(f"💾 Saving model to {model_path}...")
        torch.save({'model_state_dict': model.state_dict(), 'history': history}, model_path)
        print("✅ Training complete and model saved.")

    return model, history

print("✅ Unified Model Manager & Trainer defined!")

✅ Unified Model Manager & Trainer defined!


## Fungsi Kuantisasi Modular

Standard & Fine-Grained Quantization Ternary


In [13]:
import torch
import torch.nn as nn
import copy
import numpy as np

def standard_ternary_quantize(weight_tensor):
    original_shape = weight_tensor.shape
    flat_weights = weight_tensor.flatten()
    abs_weights = torch.abs(flat_weights)
    if abs_weights.max().item() == 0:
        return torch.zeros_like(weight_tensor), 0.0
    delta = 0.7 * torch.mean(abs_weights)
    mask = abs_weights > delta
    if mask.sum() == 0:
        return torch.zeros_like(weight_tensor), 0.0
    alpha = abs_weights[mask].sum() / mask.sum().float()
    ternary_flat = torch.zeros_like(flat_weights)
    ternary_flat[flat_weights > delta] = alpha
    ternary_flat[flat_weights < -delta] = -alpha
    return ternary_flat.reshape(original_shape), alpha.item()

def apply_standard_ternary_to_model(model, verbose=False):
    model_q = copy.deepcopy(model)
    layer_alphas = []
    for name, module in model_q.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            ternary_weight, alpha = standard_ternary_quantize(module.weight.data)
            module.weight.data = ternary_weight
            layer_alphas.append(alpha)
            if verbose: print(f"  ✓ {name}: quantized with ́={alpha:.4f}")
    return model_q, layer_alphas

def ternary_quantize_group(weight_tensor, group_size=4, asymmetric=False):
    original_shape = weight_tensor.shape
    flat_weights = weight_tensor.flatten()
    n = flat_weights.numel()
    ternary_flat = torch.zeros_like(flat_weights)
    alphas = []

    for i in range(0, n, group_size):
        group = flat_weights[i:min(i+group_size, n)]
        if len(group) == 0: continue

        if not asymmetric:
            abs_group = torch.abs(group)
            if abs_group.max().item() == 0:
                alphas.append(0.0)
                continue
            thresholds = torch.linspace(abs_group.min().item(), abs_group.max().item(), steps=20)
            best_delta, best_score = 0, -float('inf')
            for delta in thresholds:
                mask = abs_group > delta
                if mask.sum() == 0: continue
                score = (abs_group[mask].sum() ** 2) / mask.sum().float()
                if score > best_score: best_score, best_delta = score, delta

            mask = abs_group > best_delta
            if mask.sum() == 0:
                alphas.append(0.0)
                continue
            alpha = abs_group[mask].sum() / mask.sum().float()
            alphas.append(alpha.item())
            ternary_group = torch.zeros_like(group)
            ternary_group[group > best_delta] = alpha
            ternary_group[group < -best_delta] = -alpha
        else:
            # Asymmetric logic: Separate delta and alpha for positive and negative values
            pos_mask = group > 0
            neg_mask = group < 0

            a_pos, a_neg = 0.0, 0.0
            d_pos, d_neg = 0.0, 0.0

            if pos_mask.any():
                pos_vals = group[pos_mask]
                d_pos = 0.7 * torch.mean(pos_vals)
                m_pos = pos_vals > d_pos
                if m_pos.any(): a_pos = torch.mean(pos_vals[m_pos])

            if neg_mask.any():
                neg_vals = group[neg_mask]
                d_neg = 0.7 * torch.mean(torch.abs(neg_vals))
                m_neg = torch.abs(neg_vals) > d_neg
                if m_neg.any(): a_neg = torch.mean(neg_vals[m_neg])

            ternary_group = torch.zeros_like(group)
            ternary_group[group > d_pos] = a_pos
            ternary_group[group < -d_neg] = a_neg
            alphas.append((abs(a_pos) + abs(a_neg)).item() / 2.0 if torch.is_tensor(a_pos) else (abs(a_pos) + abs(a_neg)) / 2.0)

        ternary_flat[i:min(i+group_size, n)] = ternary_group

    return ternary_flat.reshape(original_shape), alphas

def apply_fgq_to_model(model, group_size=4, asymmetric=False, verbose=False):
    if isinstance(group_size, (list, set, tuple)):
        results = {}
        for gs in sorted(list(group_size)):
            if verbose: print(f"\n--- Processing Group Size: {gs} (Asymmetric={asymmetric}) ---")
            m_q, a_m = apply_fgq_to_model(model, gs, asymmetric, verbose)
            results[gs] = (m_q, a_m)
        return results

    model_q = copy.deepcopy(model)
    all_alpha_means = []
    for name, module in model_q.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            ternary_weight, alphas = ternary_quantize_group(module.weight.data, group_size, asymmetric)
            module.weight.data = ternary_weight
            m_alpha = np.mean(alphas) if alphas else 0
            all_alpha_means.append(m_alpha)
            if verbose: print(f"  ✓ {name}: FGQ N={group_size}, Asym={asymmetric}, mean_́={m_alpha:.4f}")
    return model_q, all_alpha_means

def find_best_fgq_model(fgq_results, test_loader, device):
    best_acc = -1.0
    best_gs = None
    best_model = None

    print("\n--- Evaluating All Group Sizes ---")
    for gs, (model_q, _) in fgq_results.items():
        acc = evaluate_accuracy(model_q, test_loader, device)
        print(f"Group Size {gs}: Accuracy = {acc:.2f}%")
        if acc > best_acc:
            best_acc = acc
            best_gs = gs
            best_model = model_q

    print(f"\n✅ Best Result: Group Size {best_gs} with Accuracy {best_acc:.2f}%")
    return best_gs, best_model, best_acc

## Definisi Model

### LeNet-5

In [14]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Redefine Model Class
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = nn.functional.avg_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = nn.functional.avg_pool2d(x, 2)
        x = x.view(-1, 16 * 5 * 5)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 2. Re-initialize DataLoader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Standard normalization for MNIST
])
test_dataset = datasets.MNIST(root='./data_mnist', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 3. Instantiate model and define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet5().to(device)

print("✅ Setup for verification complete: Model and Test Loader are ready.")

✅ Setup for verification complete: Model and Test Loader are ready.


### ResNet-18

In [15]:
import torch
import torch.nn as nn
from torchvision import models

# 1. Redefine Model Class for ResNet18
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet18, self).__init__()
        # Load a pre-trained ResNet18 model
        self.model = models.resnet18(weights=None) # weights=None to avoid downloading ImageNet weights for now

        # Modify the first convolution layer for single-channel input (MNIST images are grayscale)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # Modify the final fully connected layer to match the number of classes (MNIST has 10 classes)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

# Instantiate model and define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet18().to(device)

print("✅ ResNet18 model defined and ready for MNIST.")

✅ ResNet18 model defined and ready for MNIST.


# **PENGUJIAN TRAINING & KUANTISASI**

---



## Model Setup

#### ResNet-18

In [16]:
# 1. Siapkan environment (Drive, Path, Data)
model_path_resnet, train_loader_resnet, test_loader_resnet = setup_mnist_environment('resnet18_mnist_baseline.pth')

# 2. Panggil manager model
model_fp32_resnet, history_resnet = load_and_prepare_model(
    ResNet18,
    model_path_resnet,
    train_loader_resnet,
    test_loader_resnet,
    device,
    num_epochs=5
)

print(f"\nProses selesai. Akurasi Baseline: {history_resnet['accuracy'][-1]:.2f}%")

Mounted at /content/drive
❌ No model found at /content/drive/MyDrive/model_modular/resnet18_mnist_baseline.pth. Starting training...


Epoch 1/5: 100%|██████████| 469/469 [13:10<00:00,  1.69s/it]


Epoch 1: Loss = 0.1253, Accuracy = 97.23%


Epoch 2/5: 100%|██████████| 469/469 [12:58<00:00,  1.66s/it]


Epoch 2: Loss = 0.0522, Accuracy = 98.54%


...

Epoch 3: Loss = 0.0396, Accuracy = 98.37%


Epoch 4/5: 100%|██████████| 469/469 [13:04<00:00,  1.67s/it]


Epoch 4: Loss = 0.0340, Accuracy = 98.64%


Epoch 5/5: 100%|██████████| 469/469 [13:09<00:00,  1.68s/it]


Epoch 5: Loss = 0.0289, Accuracy = 98.82%
💾 Saving model to /content/drive/MyDrive/model_modular/resnet18_mnist_baseline.pth...
✅ Training complete and model saved.

Proses selesai. Akurasi Baseline: 98.82%


#### LeNet-5

In [ ]:
# 1. Siapkan environment (Drive, Path, Data)
model_path_lenet, train_loader_lenet, test_loader_lenet = setup_mnist_environment('lenet5_mnist_baseline.pth')

# 2. Panggil manager model
model_fp32_lenet, history_lenet = load_and_prepare_model(
    LeNet5,
    model_path_lenet,
    train_loader_lenet,
    test_loader_lenet,
    device,
    num_epochs=5
)

print(f"\nProses selesai. Akurasi Baseline: {history_lenet['accuracy'][-1]:.2f}%")

Mounted at /content/drive
❌ No model found at /content/drive/MyDrive/model_modular/lenet5_mnist_baseline.pth. Starting training...


Epoch 1/5: 100%|██████████| 469/469 [00:31<00:00, 14.76it/s]


Epoch 1: Loss = 0.3677, Accuracy = 96.80%


Epoch 2/5: 100%|██████████| 469/469 [00:33<00:00, 14.15it/s]


Epoch 2: Loss = 0.0913, Accuracy = 98.05%


...

Epoch 3: Loss = 0.0636, Accuracy = 98.26%


Epoch 4/5: 100%|██████████| 469/469 [00:36<00:00, 12.74it/s]


Epoch 4: Loss = 0.0490, Accuracy = 98.74%


Epoch 5/5: 100%|██████████| 469/469 [00:33<00:00, 14.05it/s]


Epoch 5: Loss = 0.0393, Accuracy = 98.89%
💾 Saving model to /content/drive/MyDrive/model_modular/lenet5_mnist_baseline.pth...
✅ Training complete and model saved.

Proses selesai. Akurasi Baseline: 98.89%


## Kuantisasi Model

### *Standard Ternary Quantization*

#### ResNet-18

In [17]:
model_ternary_resnet, alphas_resnet = apply_standard_ternary_to_model(model_fp32_resnet, verbose=True)
acc_ternary_resnet = evaluate_accuracy(model_ternary_resnet, test_loader, device)

print(f"Baseline: {history_resnet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_resnet:.2f}%")

  ✓ model.conv1: quantized with ́=0.1045
  ✓ model.layer1.0.conv1: quantized with ́=0.0786
  ✓ model.layer1.0.conv2: quantized with ́=0.0778
  ✓ model.layer1.1.conv1: quantized with ́=0.0778
  ✓ model.layer1.1.conv2: quantized with ́=0.0775
  ✓ model.layer2.0.conv1: quantized with ́=0.0577
  ✓ model.layer2.0.conv2: quantized with ́=0.0572
  ✓ model.layer2.0.downsample.0: quantized with ́=0.1521
  ✓ model.layer2.1.conv1: quantized with ́=0.0559
  ✓ model.layer2.1.conv2: quantized with ́=0.0554
  ✓ model.layer3.0.conv1: quantized with ́=0.0407
  ✓ model.layer3.0.conv2: quantized with ́=0.0386
  ✓ model.layer3.0.downsample.0: quantized with ́=0.1097
  ✓ model.layer3.1.conv1: quantized with ́=0.0368
  ✓ model.layer3.1.conv2: quantized with ́=0.0369
  ✓ model.layer4.0.conv1: quantized with ́=0.0271
  ✓ model.layer4.0.conv2: quantized with ́=0.0255
  ✓ model.layer4.0.downsample.0: quantized with ́=0.0779
  ✓ model.layer4.1.conv1: quantized with ́=0.0252
  ✓ model.layer4.1.conv2: quantized wi

#### LeNet-5

In [ ]:
model_ternary_lenet, alphas_lenet = apply_standard_ternary_to_model(model_fp32_lenet, verbose=True)
acc_ternary_lenet = evaluate_accuracy(model_ternary_lenet, test_loader, device)

print(f"Baseline: {history_lenet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_lenet:.2f}%")

  ✓ conv1: quantized with ́=0.2316
  ✓ conv2: quantized with ́=0.1280
  ✓ fc1: quantized with ́=0.0676
  ✓ fc2: quantized with ́=0.0841
  ✓ fc3: quantized with ́=0.1122
Baseline: 98.89%
Ternary: 94.18%


### *Fine-Grained Ternary Quantization*

#### ResNet-18

In [18]:
fgq_results_resnet = apply_fgq_to_model(model_fp32_resnet, group_size={4, 8, 16}, asymmetric=True, verbose=True)

best_gs_resnet, best_model_fgq_resnet, best_acc_resnet = find_best_fgq_model(fgq_results_resnet, test_loader_resnet, device)

print(f"\nBaseline Accuracy: {history_resnet['accuracy'][-1]:.2f}%")
print(f"Akurasi Terbaik ditemukan pada Group Size {best_gs_resnet}: {best_acc_resnet:.2f}%")


--- Processing Group Size: 4 (Asymmetric=True) ---
  ✓ model.conv1: FGQ N=4, Asym=True, mean_́=0.0855
  ✓ model.layer1.0.conv1: FGQ N=4, Asym=True, mean_́=0.0628
  ✓ model.layer1.0.conv2: FGQ N=4, Asym=True, mean_́=0.0619
  ✓ model.layer1.1.conv1: FGQ N=4, Asym=True, mean_́=0.0621
  ✓ model.layer1.1.conv2: FGQ N=4, Asym=True, mean_́=0.0619
  ✓ model.layer2.0.conv1: FGQ N=4, Asym=True, mean_́=0.0458
  ✓ model.layer2.0.conv2: FGQ N=4, Asym=True, mean_́=0.0464
  ✓ model.layer2.0.downsample.0: FGQ N=4, Asym=True, mean_́=0.1237
  ✓ model.layer2.1.conv1: FGQ N=4, Asym=True, mean_́=0.0453
  ✓ model.layer2.1.conv2: FGQ N=4, Asym=True, mean_́=0.0448
  ✓ model.layer3.0.conv1: FGQ N=4, Asym=True, mean_́=0.0328
  ✓ model.layer3.0.conv2: FGQ N=4, Asym=True, mean_́=0.0314
  ✓ model.layer3.0.downsample.0: FGQ N=4, Asym=True, mean_́=0.0889
  ✓ model.layer3.1.conv1: FGQ N=4, Asym=True, mean_́=0.0300
  ✓ model.layer3.1.conv2: FGQ N=4, Asym=True, mean_́=0.0301
  ✓ model.layer4.0.conv1: FGQ N=4, Asym=Tru

#### LeNet-5

In [ ]:
fgq_results_lenet = apply_fgq_to_model(model_fp32_lenet, group_size={4, 8, 16}, asymmetric=True, verbose=True)

best_gs_lenet, best_model_fgq_lenet, best_acc_lenet = find_best_fgq_model(fgq_results_lenet, test_loader, device)

print(f"\nBaseline Accuracy: {history_lenet['accuracy'][-1]:.2f}%")
print(f"Akurasi Terbaik ditemukan pada Group Size {best_gs_lenet}: {best_acc_lenet:.2f}%")


--- Processing Group Size: 4 (Asymmetric=True) ---
  ✓ conv1: FGQ N=4, Asym=True, mean_́=0.1777
  ✓ conv2: FGQ N=4, Asym=True, mean_́=0.0854
  ✓ fc1: FGQ N=4, Asym=True, mean_́=0.0496
  ✓ fc2: FGQ N=4, Asym=True, mean_́=0.0678
  ✓ fc3: FGQ N=4, Asym=True, mean_́=0.0906

--- Processing Group Size: 8 (Asymmetric=True) ---
  ✓ conv1: FGQ N=8, Asym=True, mean_́=0.1959
  ✓ conv2: FGQ N=8, Asym=True, mean_́=0.1003
  ✓ fc1: FGQ N=8, Asym=True, mean_́=0.0590
  ✓ fc2: FGQ N=8, Asym=True, mean_́=0.0791
  ✓ fc3: FGQ N=8, Asym=True, mean_́=0.1062

--- Processing Group Size: 16 (Asymmetric=True) ---
  ✓ conv1: FGQ N=16, Asym=True, mean_́=0.2127
  ✓ conv2: FGQ N=16, Asym=True, mean_́=0.1123
  ✓ fc1: FGQ N=16, Asym=True, mean_́=0.0637
  ✓ fc2: FGQ N=16, Asym=True, mean_́=0.0811
  ✓ fc3: FGQ N=16, Asym=True, mean_́=0.1095

--- Evaluating All Group Sizes ---
Group Size 4: Accuracy = 98.69%
Group Size 8: Accuracy = 98.43%
Group Size 16: Accuracy = 98.04%

✅ Best Result: Group Size 4 with Accuracy 98.69

## Perbandingan Akurasi

### ResNet18

In [19]:
print(f"Baseline: {history_resnet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_resnet:.2f}%")
print(f"FGQ best Group Size: {best_acc_resnet:.2f}%")

Baseline: 98.82%
Ternary: 23.41%
FGQ best Group Size: 98.02%


### Lenet5

In [ ]:
print(f"Baseline: {history_lenet['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary_lenet:.2f}%")
print(f"FGQ best Group Size: {best_acc_lenet:.2f}%")

Baseline: 98.89%
Ternary: 94.18%
FGQ best Group Size: 98.69%


# Ringkasan & Panduan Penggunaan Modular

Setelah melakukan refaktorisasi, eksperimen kuantisasi kini jauh lebih bersih dan terorganisir. Berikut adalah manfaat utama dari struktur baru ini:

1.  **Independensi Model**: Fungsi `apply_...` menggunakan `copy.deepcopy()`, sehingga model asli (FP32) tetap aman dan bisa digunakan berkali-kali untuk skenario berbeda.
2.  **Otomatisasi Training**: Fungsi `load_and_prepare_model` menangani logika pemuatan file dan training secara internal.
3.  **Kemudahan Eksperimen**: Membandingkan teknik kuantisasi kini hanya membutuhkan beberapa baris kode.

#### Contoh Penggunaan:

```python
# 1. Load/Train Baseline Model
model_fp32, history = load_and_prepare_model(LeNet5, model_path, train_loader, test_loader, device)

# 2. Jalankan Standard Ternary
model_ternary, alphas = apply_standard_ternary_to_model(model_fp32, verbose=True)
acc_ternary = evaluate_accuracy(model_ternary, test_loader, device)

# 3. Jalankan FGQ dengan N=4
model_fgq, mean_alphas = apply_fgq_to_model(model_fp32, group_size=4, verbose=True)
acc_fgq = evaluate_accuracy(model_fgq, test_loader, device)

print(f"Baseline: {history['accuracy'][-1]:.2f}%")
print(f"Ternary: {acc_ternary:.2f}%")
print(f"FGQ (N=4): {acc_fgq:.2f}%")
```